
## 1. 설정 (Configuration)
Kafka 접속 정보 및 Bronze 경로 설정

In [0]:
import os
import sys

SCOPE = "kv-sense-team4"

# ── Widget 입력창 추가 ──────────────────────────────────
dbutils.widgets.text("kafka_topic", "wikipedia.content.sse.raw", "Kafka 토픽")
KAFKA_TOPIC = dbutils.widgets.get("kafka_topic")

# 토픽명에서 자동으로 경로 추출
# wikipedia.content.sse.raw → wikipedia_content
parts = KAFKA_TOPIC.split(".")
TENANT_NAME = f"{parts[0]}_{parts[1]}"

# ── ADLS 인증 ───────────────────────────────────────────
adls_client_id     = dbutils.secrets.get(SCOPE, "adls-client-id")
adls_client_secret = dbutils.secrets.get(SCOPE, "adls-client-secret")
adls_tenant_id     = dbutils.secrets.get(SCOPE, "adls-tenant-id")

spark.conf.set(
    "fs.azure.account.auth.type.datacopsadls.dfs.core.windows.net", "OAuth"
)
spark.conf.set(
    "fs.azure.account.oauth.provider.type.datacopsadls.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    "fs.azure.account.oauth2.client.id.datacopsadls.dfs.core.windows.net",
    adls_client_id
)
spark.conf.set(
    "fs.azure.account.oauth2.client.secret.datacopsadls.dfs.core.windows.net",
    adls_client_secret
)
spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.datacopsadls.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token"
)

try:
    dbutils.fs.ls("abfss://bronze@datacopsadls.dfs.core.windows.net/")
    print("[OK] ADLS 접근 성공")
except Exception as e:
    print(f"[ERROR] {e}")

# ── 경로 자동 설정 ──────────────────────────────────────
KAFKA_SERVERS = "20.196.253.10:9094,20.214.71.63:9094,20.249.121.202:9094"
BRONZE_PATH   = f"abfss://bronze@datacopsadls.dfs.core.windows.net/{TENANT_NAME}/"
CHECKPOINT    = f"abfss://bronze@datacopsadls.dfs.core.windows.net/_checkpoints/{TENANT_NAME}/"

JAAS = (
    'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
    f'username="{dbutils.secrets.get(SCOPE, "kafka-username")}" '
    f'password="{dbutils.secrets.get(SCOPE, "kafka-password")}";'
)

print("[OK] 설정 완료")
print(f"[INFO] 토픽:     {KAFKA_TOPIC}")
print(f"[INFO] 테넌트:   {TENANT_NAME}")
print(f"[INFO] 저장경로: {BRONZE_PATH}")
print(f"[INFO] 체크포인트: {CHECKPOINT}")


## 2. Kafka 스트림 읽기

In [0]:
df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_PLAINTEXT")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", JAAS)
    .option("failOnDataLoss", "false")
    .load()
)

print("Kafka 스트림 연결 완료")
df_raw.printSchema()


## 3. JSON 파싱 + 원본 전체 보존

In [0]:
from pyspark.sql.functions import (
    col, from_json, current_timestamp, lit
)
from pyspark.sql.types import StringType

# Kafka value를 문자열로 변환 (_kafka_topic 컬럼 제거 — 폴더명으로 도메인 구분)
df_parsed = (
    df_raw
    .select(
        col("value").cast("string").alias("raw_json"),
        col("timestamp").alias("kafka_timestamp"),
    )
    .withColumn("_bronze_loaded_at", current_timestamp())
    .withColumn("_source", from_json(
        col("raw_json"), "map<string,string>"
    ))
)

print("파싱 완료")
df_parsed.printSchema()


## 04 Bronze Delta Table 적재


In [0]:
query = (
    df_parsed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(processingTime="10 seconds")
    .start(BRONZE_PATH)
)

print(f"✅ 스트리밍 시작!")
print(f"저장 경로: {BRONZE_PATH}")
print(f"상태: {query.status}")


## 05 적재 확인 
4 셀 실행 후 30초 후에 실행

In [0]:
import time
time.sleep(30)

df_check = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze 적재 건수: {df_check.count()}")
df_check.show(3, truncate=True)